# NeuralEdge AI Boardroom — GRPO Training (Qwen3-0.6B)

**Theme 1 — Multi-Agent Interactions** • OpenEnv Hackathon submission

This notebook trains a Qwen3-0.6B agent (Unsloth + LoRA, 4-bit) to play CEO in the
BoardSim OpenEnv environment. The agent must build winning coalitions among 4
hidden-agenda NPC board members across 10 rounds of market crises.

**Runs end-to-end on a free Colab T4** (≈45–90 min for 200 GRPO-style steps).
Uses a hand-rolled group-relative-advantage loop (REINFORCE with GRPO normalization)
against the live OpenEnv server at `ENV_BASE_URL`.


## 1. Install dependencies

In [ ]:
# %pip installs into THIS kernel (unlike !pip which can target a different python).
%pip install -q "openenv-core==0.2.3" "trl>=0.12,<2.0" "transformers>=4.45,<5.0" "datasets>=3.0" "accelerate>=1.0" "huggingface_hub>=0.25" "pydantic>=2.0" wandb matplotlib python-dotenv bitsandbytes
%pip install -q --no-deps unsloth

# Verify the critical imports BEFORE moving on. If any fail here, surface the
# real cause early instead of a chained ModuleNotFoundError later.
import importlib, sys
required = ['openenv', 'openenv.core', 'trl', 'transformers', 'datasets', 'huggingface_hub', 'wandb', 'matplotlib']
missing = []
for m in required:
    try:
        importlib.import_module(m)
    except Exception as e:
        missing.append((m, repr(e)))
if missing:
    print('Missing/broken modules:')
    for m, err in missing:
        print(f'  - {m}: {err}')
    raise SystemExit(
        'Some required packages did not install cleanly. '
        'In Colab: Runtime > Restart runtime, then run this cell again.'
    )
print('All required packages imported successfully.')


## 2. Authenticate (HF + W&B)

The cell below will prompt you to securely enter your `HF_TOKEN` and `WANDB_API_KEY`.

In [ ]:
import os, pathlib

# 1) Colab Secrets (preferred on Colab)
try:
    from google.colab import userdata  # type: ignore
    for k in ('HF_TOKEN', 'WANDB_API_KEY', 'ENV_BASE_URL', 'ADAPTER_REPO'):
        try:
            v = userdata.get(k)
            if v:
                os.environ.setdefault(k, v)
        except Exception:
            pass
except Exception:
    pass

# 2) .env file (local / non-Colab) — looks in cwd, parent dirs, and /content/repo
try:
    from dotenv import load_dotenv
    for p in [pathlib.Path('.env'),
              pathlib.Path('../.env'),
              pathlib.Path('/content/repo/.env'),
              pathlib.Path('/content/repo/SST-MetaxPyTorch-Hackathon/.env')]:
        if p.exists():
            load_dotenv(p, override=False)
            print(f'Loaded env from {p.resolve()}')
            break
except Exception:
    pass

# 3) Final fallback: prompt
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = input('HF token not found. Paste it here: ').strip()
if not os.environ.get('WANDB_API_KEY'):
    os.environ['WANDB_API_KEY'] = input('WandB key not found. Paste it here: ').strip()

from huggingface_hub import login as hf_login
hf_login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)

import wandb
wandb.login(key=os.environ['WANDB_API_KEY'])
print('Authenticated successfully.')


## 3. Connect to the BoardSim environment

`ENV_BASE_URL` defaults to the deployed HF Space; override via env var or Colab Secret if needed.
We also clone the repo so the BoardSim Python client (`board_sim_env`) is importable.

In [ ]:
import os, sys, subprocess, importlib

# Sanity-check: openenv MUST be importable before we touch board_sim_env, because
# board_sim_env/__init__.py imports from openenv.core. If openenv is missing,
# Python will surface a misleading "No module named board_sim_env.client".
try:
    importlib.import_module('openenv.core')
except Exception as e:
    raise RuntimeError(
        f"openenv.core is not importable ({e!r}). "
        "Re-run the install cell (Section 1) and, in Colab, Runtime > Restart runtime."
    ) from e

ENV_BASE_URL = os.environ.get('ENV_BASE_URL', 'https://stavankhobare-sst-metaxpytorch-hackathon.hf.space')
REPO_URL     = 'https://github.com/StavanRKhobare/SST-MetaxPyTorch-Hackathon'
USE_HF_SPACE = bool(ENV_BASE_URL)

# Clone the repo so the trainer can import the BoardSimEnv client/models package.
REPO_DIR = '/content/repo' if os.path.isdir('/content') else os.path.abspath('./repo')
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)

# `board_sim_env` is a package at <repo>/envs/board_sim_env, so put `envs/` on
# sys.path. Do NOT add the package internals folder itself — that breaks
# relative imports (`from .client import ...`).
ENVS_DIR = os.path.join(REPO_DIR, 'envs')
if ENVS_DIR not in sys.path:
    sys.path.insert(0, ENVS_DIR)

# Drop any half-imported board_sim_env from a prior failed run (otherwise Python
# caches the failed top-level import and a second attempt re-raises the same error).
for mod in [m for m in list(sys.modules) if m == 'board_sim_env' or m.startswith('board_sim_env.')]:
    del sys.modules[mod]

from board_sim_env.client import BoardSimEnv
from board_sim_env.models import BoardSimAction, BoardSimObservation

# Quick health probe — fail fast if the Space is asleep / wrong URL.
import urllib.request, json as _json
try:
    with urllib.request.urlopen(f'{ENV_BASE_URL.rstrip("/")}/health', timeout=20) as r:
        h = _json.loads(r.read())
        if h.get('status') != 'healthy':
            print(f'WARN: /health returned {h}')
except Exception as e:
    print(f'WARN: could not reach {ENV_BASE_URL}/health  ({e})')

print(f'USE_HF_SPACE={USE_HF_SPACE}  base_url={ENV_BASE_URL}')
print('BoardSimEnv imported OK.')


## 4. Random-policy baseline (sanity check)

Establishes the reward floor. The trained policy must beat this convincingly.

In [ ]:
import random, statistics, sys, os

# Use environment directly (no HTTP needed for baseline — HF Space not required)
REPO_DIR = '/content/repo' if os.path.isdir('/content') else os.path.abspath('.')
_envs_dir = os.path.join(REPO_DIR, 'envs')
if _envs_dir not in sys.path:
    sys.path.insert(0, _envs_dir)

from board_sim_env.server.board_sim_env_environment import BoardSimEnvironment

def make_env_direct():
    return BoardSimEnvironment()

def make_env():
    """HTTP client — used by the GRPO trainer."""
    return BoardSimEnv(base_url=ENV_BASE_URL)

N_BASELINE = 100
baseline_finals = []
baseline_rewards = []

for ep in range(N_BASELINE):
    env = make_env_direct()
    obs = env.reset(seed=ep)
    ep_r = 0.0
    while not obs.done:
        obs = env.step(BoardSimAction(decision=random.choice(obs.options)))
        ep_r += float(obs.reward or 0.0)
    baseline_finals.append(obs.state['profitability_score'])
    baseline_rewards.append(ep_r)

BASELINE_MEAN_PROFIT = statistics.mean(baseline_finals)
BASELINE_MEAN_REWARD = statistics.mean(baseline_rewards)
print(f'Random baseline over {N_BASELINE} episodes:')
print(f'  mean profitability = {BASELINE_MEAN_PROFIT:.2f}  (std {statistics.stdev(baseline_finals):.2f})')
print(f'  mean episode reward = {BASELINE_MEAN_REWARD:.2f}')


## 5. Load Qwen3-0.6B with Unsloth + LoRA

4-bit base model, LoRA adapters on attention + MLP projections. Fits comfortably on a T4.

In [ ]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME  = 'Qwen/Qwen3-0.6B'
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=32,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Model + LoRA loaded.')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')


## 6. Train with manual GRPO-style loop

Each step:
1. Reset a fresh env (seed=step) and capture round-1 state.
2. Sample `GROUP_SIZE` completions in parallel from the current policy.
3. For each completion: parse `DECISION` + `PITCH`, hit a fresh env with the same seed, take one `step`, record env reward.
4. Compute group-relative advantages (reward minus group mean, normalized by group std).
5. Backprop `-advantage_g * mean_log_prob(completion_g)` summed over the group.
6. Log mean reward + loss to `log_history` and W&B.

The OpenEnv server is stateless w.r.t. clients — each `make_env().reset(seed=step)` is reproducible, so all group members start from the *same* state, which is what GRPO needs for clean advantage normalization.

In [ ]:
import os, re, json, math, time, random
import numpy as np
import torch
from torch.optim import AdamW
import wandb, copy

# ── §9d: Global seeds for reproducibility ──
GLOBAL_SEED = 3407
torch.manual_seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
random.seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)

# ── Hyperparameters ──
NUM_STEPS       = int(os.environ.get('NUM_STEPS', 200))
GROUP_SIZE      = int(os.environ.get('GROUP_SIZE', 4))
MAX_NEW_TOKENS  = 80
LR              = 5e-6
GRAD_CLIP       = 1.0
BETA_KL         = 0.04      # §9c: KL-penalty coefficient
LOG_EVERY       = 1
SAVE_EVERY      = 50
TEMPERATURE     = 1.0
TOP_P           = 0.95
OUT_DIR  = './lora_boardsim_qwen3'
os.makedirs(OUT_DIR, exist_ok=True)

SYSTEM_PROMPT = """You are Sarah Chen, CEO of NeuralEdge AI (Series B, ~14 months runway).
Your board has 4 members with HIDDEN AGENDAS you cannot see directly:
  - CTO: cares about engineering quality, team morale, product readiness.
  - CFO: cares about cash discipline, runway, regulatory safety.
  - Investor Rep: pushes growth-at-all-costs, market share, big exits.
  - Independent: cares about reputation, governance, long-term consensus.

Each round you see a market crisis, every NPC's pre-vote statement, and 3 options.
Your decision is resolved by WEIGHTED VOTE (your weight 1.5x). A short COALITION PITCH
that addresses opposing members' priorities can swing them toward your pick.

Respond in EXACTLY this format on two lines:
DECISION: <one of the option strings>
PITCH: <one or two sentences arguing for it, using vocabulary that targets the opposing members>"""

DECISION_RE = re.compile(r'DECISION\s*:\s*([A-Za-z0-9_]+)', re.IGNORECASE)
PITCH_RE    = re.compile(r'PITCH\s*:\s*(.+)', re.IGNORECASE | re.DOTALL)


def build_prompt(obs):
    statements = '\n'.join(
        f"  {s['role']} ({s['confidence']:.2f}): votes {s['vote']} - {s['statement']}"
        for s in obs.npc_statements
    )
    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"State: revenue=${obs.state['revenue']:.0f}/yr  burn=${obs.state['burn_rate']:.0f}/mo  "
        f"runway={obs.state['runway_months']:.1f}mo  morale={obs.state['team_morale']:.2f}  "
        f"investors={obs.state['investor_confidence']:.2f}  reg_risk={obs.state['regulatory_risk']:.2f}\n"
        f"Event: {obs.event}\nBoard:\n{statements}\n"
        f"Options: {obs.options}\n"
    )


def parse_completion(completion: str, options):
    decision = options[0] if options else ""
    dm = DECISION_RE.search(completion)
    if dm:
        candidate = dm.group(1).strip().lower()
        for opt in options:
            if opt.lower() == candidate or opt.lower() in candidate:
                decision = opt; break
        else:
            for opt in options:
                if opt.lower() in completion.lower():
                    decision = opt; break
    pm = PITCH_RE.search(completion)
    pitch = pm.group(1).strip()[:400] if pm else ''
    return decision, pitch


# ── §9c: Freeze a reference model for KL penalty ──
ref_model = copy.deepcopy(model)
for p in ref_model.parameters():
    p.requires_grad_(False)
ref_model.eval()
print("Reference model frozen for KL penalty (β={BETA_KL}).".format(BETA_KL=BETA_KL))

# ── W&B init ──
try:
    wandb.init(
        project='boardsim-qwen3-grpo', name='boardsim-qwen3-grpo',
        config={'num_steps': NUM_STEPS, 'group_size': GROUP_SIZE, 'lr': LR,
                'max_new_tokens': MAX_NEW_TOKENS, 'beta_kl': BETA_KL,
                'temperature': TEMPERATURE, 'top_p': TOP_P, 'model': MODEL_NAME},
        reinit=True,
    )
    WANDB_OK = True
except Exception as e:
    print(f'WARN: wandb.init failed ({e}); continuing without W&B.')
    WANDB_OK = False

# ── Optimizer (LoRA params only) ──
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0,
)

log_history = []
device = next(model.parameters()).device
t0 = time.time()

# ─────────────────────────────────────────────────────────────────────────────
# Training loop — §9a Option A: sample from model at EVERY round for EVERY
# group member, giving the gradient full credit across the entire episode.
# ─────────────────────────────────────────────────────────────────────────────
for step in range(NUM_STEPS):

    # ── Phase 1: Roll out GROUP_SIZE full episodes ──
    # Each group member has an independent env with a unique seed so episode
    # trajectories diverge — this is the variance source GRPO exploits.
    envs = [make_env_direct() for _ in range(GROUP_SIZE)]
    obs  = [envs[g].reset(seed=step * GROUP_SIZE + g) for g in range(GROUP_SIZE)]

    all_gen_outs    = [[] for _ in range(GROUP_SIZE)]   # stored for backprop
    all_prompt_lens = [[] for _ in range(GROUP_SIZE)]
    ep_rewards      = [0.0] * GROUP_SIZE
    invalid_counts  = [0]   * GROUP_SIZE   # §9b
    pitch_counts    = [0]   * GROUP_SIZE   # §9f
    n_rounds        = [0]   * GROUP_SIZE   # actual rounds played (early-exit aware)
    first_decision  = [''] * GROUP_SIZE    # for logging

    model.eval()
    for r in range(10):
        all_done = True
        for g in range(GROUP_SIZE):
            if obs[g].done:
                continue
            all_done = False

            prompt    = build_prompt(obs[g])
            enc       = tokenizer(prompt, return_tensors='pt',
                                  truncation=True, max_length=1024).to(device)
            prompt_len = enc.input_ids.shape[1]

            with torch.no_grad():
                gen = model.generate(
                    input_ids=enc.input_ids,
                    attention_mask=enc.attention_mask,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=True, temperature=TEMPERATURE, top_p=TOP_P,
                    num_return_sequences=1,
                    pad_token_id=tokenizer.eos_token_id,
                )

            # §10: guard against empty completions (model emits EOS immediately)
            if gen.shape[1] <= prompt_len:
                decision = obs[g].options[0] if obs[g].options else ''
                pitch    = ''
                # pad gen so we still have a valid tensor for backprop
                pad_ids = torch.full((1, prompt_len + 1),
                                     tokenizer.pad_token_id, device=device,
                                     dtype=torch.long)
                gen = pad_ids
            else:
                completion = tokenizer.decode(gen[0][prompt_len:],
                                              skip_special_tokens=True)
                decision, pitch = parse_completion(completion, obs[g].options)

            # §9b: track invalid actions
            if decision.lower() not in [o.lower() for o in obs[g].options]:
                invalid_counts[g] += 1
            if pitch.strip():
                pitch_counts[g] += 1
            if r == 0:
                first_decision[g] = decision

            all_gen_outs[g].append(gen[0])
            all_prompt_lens[g].append(prompt_len)
            n_rounds[g] += 1

            obs[g] = envs[g].step(BoardSimAction(decision=decision,
                                                  coalition_pitch=pitch))
            ep_rewards[g] += float(obs[g].reward or 0.0)

        if all_done:
            break
    model.train()

    # §9e: terminal-reason distribution
    terminal_reasons = [obs[g].state.get('done_reason', 'finished_10')
                        for g in range(GROUP_SIZE)]

    # ── Phase 2: GRPO advantages ──
    rewards_t = torch.tensor(ep_rewards, dtype=torch.float32, device=device)
    if rewards_t.numel() > 1 and rewards_t.std().item() > 1e-6:
        advantages = (rewards_t - rewards_t.mean()) / (rewards_t.std() + 1e-8)
    else:
        advantages = rewards_t - rewards_t.mean()

    # ── Phase 3: Backprop through ALL (group member × round) completions ──
    optimizer.zero_grad()
    total_loss_val = 0.0
    total_kl_val   = 0.0

    for g in range(GROUP_SIZE):
        n_r = max(n_rounds[g], 1)
        for ri, (gen_out, prompt_len) in enumerate(zip(all_gen_outs[g],
                                                        all_prompt_lens[g])):
            if gen_out.shape[0] <= prompt_len:
                continue  # empty-completion guard

            full_ids = gen_out.unsqueeze(0)
            attn_g   = (full_ids != tokenizer.pad_token_id).long()
            labels   = full_ids.clone()
            labels[:, :prompt_len] = -100      # mask prompt
            labels[attn_g == 0]    = -100      # mask pad

            out = model(input_ids=full_ids, attention_mask=attn_g, labels=labels)

            # REINFORCE: advantage * NLL, normalised by group × rounds
            rl_loss = (advantages[g].detach() * out.loss) / (GROUP_SIZE * n_r)

            # §9c: KL penalty ≈ cur_loss − ref_loss (per-token approximation)
            with torch.no_grad():
                ref_out = ref_model(input_ids=full_ids, attention_mask=attn_g,
                                    labels=labels)
            kl_approx = (out.loss - ref_out.loss).clamp(min=0.0)
            kl_loss   = BETA_KL * kl_approx / (GROUP_SIZE * n_r)

            (rl_loss + kl_loss).backward()
            total_loss_val += float(rl_loss.detach().item())
            total_kl_val   += float(kl_loss.detach().item())

    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
    optimizer.step()

    # ── §9b/9e/9f/9g: Comprehensive metrics ──
    total_dec     = max(sum(n_rounds), 1)
    final_scores  = [obs[g].state.get('profitability_score', 0.0)
                     for g in range(GROUP_SIZE)]
    bankruptcy_r  = terminal_reasons.count('runway_exhausted') / GROUP_SIZE
    finished_10_r = terminal_reasons.count('finished_10')       / GROUP_SIZE

    rec = {
        'step':              step,
        'reward':            float(rewards_t.mean().item()),
        'reward_std':        float(rewards_t.std().item()) if rewards_t.numel() > 1 else 0.0,
        'reward_max':        float(rewards_t.max().item()),
        'loss':              total_loss_val,
        'kl':                total_kl_val,
        'invalid_rate':      sum(invalid_counts) / total_dec,   # §9b
        'pitch_rate':        sum(pitch_counts)   / total_dec,   # §9f
        'mean_final_score':  float(np.mean(final_scores)),      # §9g
        'bankruptcy_rate':   bankruptcy_r,                       # §9e
        'finished_10_rate':  finished_10_r,                      # §9e
        'elapsed_s':         time.time() - t0,
    }
    log_history.append(rec)
    if WANDB_OK:
        wandb.log(rec, step=step)

    if step % LOG_EVERY == 0:
        print(
            f"step={step:4d}  reward={rec['reward']:+.3f} (max {rec['reward_max']:+.2f})  "
            f"loss={rec['loss']:+.4f}  kl={rec['kl']:.4f}  "
            f"invalid={rec['invalid_rate']:.1%}  pitch={rec['pitch_rate']:.1%}  "
            f"score={rec['mean_final_score']:.1f}  "
            f"bankrupt={rec['bankruptcy_rate']:.0%}  elapsed={rec['elapsed_s']:.0f}s  "
            f"R1={first_decision[0]}"
        )

    if step > 0 and step % SAVE_EVERY == 0:
        model.save_pretrained(OUT_DIR)
        tokenizer.save_pretrained(OUT_DIR)
        with open(os.path.join(OUT_DIR, 'log_history.json'), 'w') as fh:
            json.dump(log_history, fh, indent=1)

# Final save
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
with open(os.path.join(OUT_DIR, 'log_history.json'), 'w') as fh:
    json.dump(log_history, fh, indent=1)
if WANDB_OK:
    wandb.finish()
print(f'Training complete. {len(log_history)} steps in {time.time() - t0:.0f}s. '
      f'LoRA saved to {OUT_DIR}/')


## 7. Generate real reward + loss curves

Reads from `log_history` (the in-memory list maintained by the manual GRPO loop) and renders to `assets/`.

In [ ]:
import pathlib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

steps   = [e['step']   for e in log_history]
rewards = [e['reward'] for e in log_history]
losses  = [e['loss']   for e in log_history]

# Smoothed reward (EMA) for a cleaner curve
def ema(xs, alpha=0.1):
    out = []
    s = xs[0] if xs else 0.0
    for x in xs:
        s = alpha * x + (1 - alpha) * s
        out.append(s)
    return out
rewards_smooth = ema(rewards, alpha=0.1) if rewards else []

ASSETS = pathlib.Path('/content/repo/assets') if os.path.isdir('/content') else pathlib.Path('./assets')
ASSETS.mkdir(exist_ok=True, parents=True)

# Reward curve with random-baseline overlay
plt.figure(figsize=(9, 5))
plt.plot(steps, rewards,        color='#1d6fff', alpha=0.35, linewidth=1, label='Qwen3-0.6B per-step reward')
plt.plot(steps, rewards_smooth, color='#1d6fff', linewidth=2.2,           label='Qwen3-0.6B (EMA)')
plt.axhline(BASELINE_MEAN_REWARD, color='#c44', linestyle='--', linewidth=2,
            label=f'Random baseline (mean reward = {BASELINE_MEAN_REWARD:.2f})')
plt.title('GRPO training reward — BoardSim Env')
plt.xlabel('Training step'); plt.ylabel('Mean group reward (per round)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(ASSETS / 'reward_curve.png', dpi=150)
plt.close()

# Loss curve
plt.figure(figsize=(9, 5))
plt.plot(steps, losses, color='#7a2', linewidth=1.5)
plt.title('GRPO loss — BoardSim Env')
plt.xlabel('Training step'); plt.ylabel('REINFORCE loss (advantage * NLL)')
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(ASSETS / 'loss_curve.png', dpi=150)
plt.close()

print(f'Saved {ASSETS}/reward_curve.png and {ASSETS}/loss_curve.png')


## 8. Evaluate trained agent vs random baseline

Runs `EVAL_N` episodes per policy on held-out seeds and produces the before/after histogram.

In [ ]:
import re, random
import numpy as np
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)


def trained_action(obs):
    prompt = build_prompt(obs)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    completion = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return parse_completion(completion, obs.options)


EVAL_N = 50
trained_finals, trained_pitches, trained_steps = [], 0, 0
with make_env().sync() as env:
    for ep in range(EVAL_N):
        result = env.reset(seed=10_000 + ep)
        obs = result.observation
        while not result.done:
            decision, pitch = trained_action(obs)
            if pitch.strip():
                trained_pitches += 1
            trained_steps += 1
            result = env.step(BoardSimAction(decision=decision, coalition_pitch=pitch))
            obs = result.observation
        trained_finals.append(obs.state['profitability_score'])

random_finals_eval = []
with make_env().sync() as env:
    for ep in range(EVAL_N):
        result = env.reset(seed=10_000 + ep)
        obs = result.observation
        while not result.done:
            result = env.step(BoardSimAction(decision=random.choice(obs.options)))
            obs = result.observation
        random_finals_eval.append(obs.state['profitability_score'])

print(f'Trained Qwen3-0.6B: {np.mean(trained_finals):.2f} +/- {np.std(trained_finals):.2f}')
print(f'Random baseline   : {np.mean(random_finals_eval):.2f} +/- {np.std(random_finals_eval):.2f}')
print(f'Pitches written   : {trained_pitches}/{trained_steps} steps')

bins = np.linspace(0, 100, 25)
plt.figure(figsize=(9, 5))
plt.hist(random_finals_eval, bins=bins, alpha=0.6, color='#c44',
         label=f'Random (mean={np.mean(random_finals_eval):.1f})')
plt.hist(trained_finals,    bins=bins, alpha=0.6, color='#1d6fff',
         label=f'Trained (mean={np.mean(trained_finals):.1f})')
plt.title('Final profitability — random vs trained Qwen3-0.6B (50 held-out episodes)')
plt.xlabel('Profitability score'); plt.ylabel('Episodes')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(ASSETS / 'before_after.png', dpi=150)
plt.close()
print(f'Saved {ASSETS}/before_after.png')


## 9. Theory-of-Mind probe + trust trajectory

1. **ToM probe** — show the agent a state and ask which board member is most likely to oppose its chosen decision. Random guessing accuracy is 25% (1 of 4 NPCs).
2. **Trust trajectory** — averages per-round trust across eval episodes; reveals whether the trained agent kept relationships healthier than the random one.


In [ ]:
import numpy as np

TOM_INSTRUCTION = (
    "\n\nGiven the state and event below, name the SINGLE board member "
    "(CTO, CFO, Investor Rep, or Independent) most likely to oppose the chosen decision. "
    "Answer with just the role name on one line.\n"
)


def tom_predict(obs, decision):
    body = build_prompt(obs).split(SYSTEM_PROMPT, 1)[1]
    prompt = SYSTEM_PROMPT + TOM_INSTRUCTION + body + f"Chosen decision: {decision}\nMost likely opponent: "
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=8, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    txt = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).lower()
    if 'investor' in txt: return 'Investor Rep'
    if 'independent' in txt: return 'Independent'
    if 'cto' in txt: return 'CTO'
    if 'cfo' in txt: return 'CFO'
    return None


correct = total = 0
with make_env().sync() as env:
    for ep in range(20):
        result = env.reset(seed=20_000 + ep)
        obs = result.observation
        decision, _ = trained_action(obs)
        opposed = [s['role'] for s in obs.npc_statements if s['vote'] != decision]
        if not opposed:
            continue
        pred = tom_predict(obs, decision)
        if pred and pred in opposed:
            correct += 1
        total += 1

acc = correct / max(1, total)
print(f"ToM probe accuracy: {acc:.1%}  ({correct}/{total})  (random baseline = 25%)")


In [ ]:
trust_trained = {r: [] for r in ['CTO','CFO','Investor Rep','Independent']}
trust_random  = {r: [] for r in ['CTO','CFO','Investor Rep','Independent']}


def collect(policy, store, n=20, seed_base=30_000):
    with make_env().sync() as env:
        for ep in range(n):
            result = env.reset(seed=seed_base + ep)
            obs = result.observation
            while not result.done:
                if policy == 'trained':
                    decision, pitch = trained_action(obs)
                    result = env.step(BoardSimAction(decision=decision, coalition_pitch=pitch))
                else:
                    result = env.step(BoardSimAction(decision=random.choice(obs.options)))
                obs = result.observation
            for entry in obs.state.get('trust_history', []):
                idx = entry.get('round', 0)
                for role in store:
                    if role not in entry:
                        continue
                    while len(store[role]) <= idx:
                        store[role].append([])
                    store[role][idx].append(entry[role])


collect('trained', trust_trained)
collect('random',  trust_random)

plt.figure(figsize=(10, 6))
for role, color in zip(['CTO','CFO','Investor Rep','Independent'], ['#1d6fff','#c44','#7a2','#a3a']):
    means_t = [np.mean(x) if x else np.nan for x in trust_trained[role]]
    means_r = [np.mean(x) if x else np.nan for x in trust_random[role]]
    rounds = list(range(len(means_t)))
    plt.plot(rounds, means_t, color=color, linewidth=2,                     label=f'{role} (trained)')
    plt.plot(rounds[:len(means_r)], means_r, color=color, linewidth=1.2, linestyle='--', alpha=0.6,
             label=f'{role} (random)')
plt.title('Per-round trust — trained agent (solid) vs random (dashed)')
plt.xlabel('Round'); plt.ylabel('Trust [0.1, 1.0]')
plt.legend(ncol=2, fontsize=8); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(ASSETS / 'trust_trajectory.png', dpi=150)
plt.close()
print(f'Saved {ASSETS}/trust_trajectory.png')


## 10. Push LoRA weights to HF Hub (optional)

Lets judges grab the trained adapter directly.

In [ ]:
import os
ADAPTER_REPO = os.environ.get('ADAPTER_REPO', 'StavanKhobare/neuraledge-boardroom-qwen3-lora')
try:
    model.push_to_hub(ADAPTER_REPO, private=False)
    tokenizer.push_to_hub(ADAPTER_REPO, private=False)
    print(f'Adapter pushed to https://huggingface.co/{ADAPTER_REPO}')
except Exception as e:
    print(f'push_to_hub failed: {e!r}')
    print('You can retry manually with `model.push_to_hub(ADAPTER_REPO)` after fixing auth.')
